In [9]:
from sentence_transformers import CrossEncoder
ce = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

In [11]:
# Semantic event matcher (SBERT) + freshness + MMR diversity + light business rules
# Features: 1) stable cache, 2) MMR on cosine, 3) TZ-aware time, 4) robust internals (no user filters) for now,
# 5) better text construction, 6) cold-start blend hooks, 7) optional cross-encoder rerank,
# 8) guardrails, 9) diversity caps, 10) observability

from pathlib import Path
from datetime import datetime, timedelta, timezone, date
import hashlib, pickle, re, math, os
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from zoneinfo import ZoneInfo

# optional cross-encoder rerank
try:
    from sentence_transformers import CrossEncoder
    _HAS_CROSS = True
except Exception:
    _HAS_CROSS = False

# CONFIGS
CSV_IN   = Path("parsed_events.csv")
CSV_OUT  = Path("recommendations.csv")
CACHE    = Path(".cache_event_embeddings.pkl")

MODEL_ID = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"  # embeddings
CROSS_ID = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"                       # optional reranker

# FIXED TIMEZONE
TZ = ZoneInfo("Europe/Berlin")

# Scoring weights
W_SIM        = 0.9   # similarity weight
W_TIME       = 0.1   # freshness weight
HALF_LIFE_DAYS = 14.0
MMR_LAMBDA     = 0.30            # 0=more diversity, 1=more relevance
TOP_K          = 12              # final number of recs to return
WINDOW         = 50              # candidate window before MMR
RERANK_TOP_N   = 40              # pass this many to cross-encoder reranker if available

# Business rules
AUTO_SKIP_PAST      = True       # ignore events strictly in the past
CAP_PER_DAY         = 2          # max events per date in final list
CAP_PER_ORGANIZER   = 2          # max events per organizer/source in final list
REQUIRE_AT_LEAST_ONE_CULTURAL = True   # try to include one cultural event if close in score
# optional: but to include at least one cultural event if none selected in the interests

CULTURE_KEYS = {
    # Music & concerts
    "concert","orchestra","choir","recital","jazz","piano","violin","symphony",
    "opera","quartet","ensemble","acoustic","jam session","chamber music",
    "klavier","konzert","chor","orchester","singen","aufführung",

    # Visual arts & design
    "exhibition","gallery","painting","sculpture","installation","artwork",
    "museum","vernissage","fotografie","ausstellung","kunst","galerie","malerei",

    # Film & cinema
    "film screening","cinema","movie","documentary","short film","animation",
    "kino","filmabend","dokumentarfilm","filmschau",

    # Theatre & performing arts
    "theatre","theater","play","drama","stage","acting","musical","ballet",
    "schauspiel","theaterstück","bühne","tanz","performance art","kabarett",

    # Literature & writing
    "poetry","reading","book presentation","literature","novel","storytelling",
    "lesung","literatur","schriftsteller","erzählen","autor","buchvorstellung",

    # Heritage & cultural studies
    "heritage","archaeology","museum tour","art history","cultural heritage",
    "denkmal","museumführung","archäologie","kulturgeschichte"
}

# 40 single-word labels with rich descriptions
INTEREST_DESCRIPTIONS = {
    # STEM & Data
    "AI": "Artificial intelligence, machine learning, deep learning, neural networks, automation, generative models, data-driven systems.",
    "Data": "Data science, big data analytics, visualization, data mining, statistics, predictive modeling, data ethics.",
    "Math": "Mathematics, probability, statistics, optimization, algebra, geometry, applied and computational math.",
    "Physics": "Physics, quantum mechanics, astrophysics, particle physics, thermodynamics, materials science.",
    "Chemistry": "Chemistry, green chemistry, sustainable chemistry, biochemistry, materials, molecular design.",
    "Biology": "Biology, genetics, ecology, molecular biology, life sciences, evolution, environmental biology.",
    "Medicine": "Medicine, healthcare, clinical research, biomedical engineering, epidemiology, public health.",
    "Engineering": "Engineering, electrical, mechanical, civil, chemical, systems design, robotics, innovation.",
    "Technology": "Technology, computing, digital innovation, software, hardware, internet, networks, IT systems.",

    # Economics, Business & Social Sciences
    "Economics": "Economics, econometrics, microeconomics, macroeconomics, trade, industrial organization, policy.",
    "Finance": "Finance, banking, investment, financial markets, risk management, asset pricing, accounting.",
    "Management": "Management, strategy, marketing, entrepreneurship, leadership, innovation, organizations.",
    "Policy": "Public policy, governance, administration, regulation, institutions, social impact, decision making.",
    "Law": "Law, legal studies, constitutional, international, data protection, intellectual property, regulation.",
    "Sociology": "Sociology, inequality, social networks, digital society, demography, urban and rural life.",
    "Psychology": "Psychology, behavior, cognition, mental health, neuroscience, learning, motivation.",
    "Education": "Education, learning, pedagogy, training, digital learning, teaching, curriculum development.",
    "Communication": "Communication, journalism, public relations, media, rhetoric, storytelling, digital media.",
    "Linguistics": "Linguistics, languages, translation, semantics, phonetics, discourse analysis, language teaching.",

    # Humanities & Culture
    "Philosophy": "Philosophy, ethics, logic, epistemology, metaphysics, philosophy of science and mind.",
    "History": "History, archaeology, cultural heritage, archives, world history, historiography.",
    "Literature": "Literature, writing, reading, poetry, creative writing, comparative literature, storytelling.",
    "Theology": "Theology, religion, faith, spirituality, interreligious dialogue, ethics, theology of culture.",
    "Arts": "Arts, visual arts, painting, sculpture, design, creativity, art history, exhibitions.",
    "Music": "Music, concerts, performances, orchestras, composition, musicology, sound art.",
    "Film": "Film, cinema, screenings, documentaries, film analysis, visual culture, media art.",
    "Theatre": "Theatre, drama, stage performance, acting, dance, performing arts.",
    "Culture": "Culture, festivals, traditions, intercultural exchange, museums, literature and art events.",

    # Cross-disciplinary & Societal Topics
    "Sustainability": "Sustainability, climate change, energy, biodiversity, environment, green transition, circular economy.",
    "Environment": "Environment, conservation, ecology, natural resources, pollution, water, ecosystems.",
    "Health": "Health, wellness, lifestyle, nutrition, sports, fitness, preventive medicine, well-being.",
    "Diversity": "Diversity, inclusion, equality, gender studies, human rights, accessibility, social justice.",
    "Ethics": "Ethics, integrity, responsibility, moral philosophy, AI ethics, bioethics, sustainability ethics.",
    "Urban": "Urban development, smart cities, architecture, housing, mobility, infrastructure.",
    "Innovation": "Innovation, creativity, entrepreneurship, startups, research transfer, new technologies.",
    "Digital": "Digital transformation, digitalization, online platforms, information society, cyber trends.",

    # Skills & Career
    "Career": "Career development, alumni events, mentoring, recruiting, professional networking, job search.",
    "Workshop": "Workshops, training sessions, bootcamps, hackathons, hands-on learning, practical skills.",
    "Seminar": "Seminars, colloquia, academic lectures, research talks, invited speakers, department events.",
    "Conference": "Conferences, symposia, congresses, large-scale research meetings, keynote talks.",
    "Networking": "Networking events, receptions, meetups, community building, collaborative gatherings.",
    "Startup": "Startups, entrepreneurship, venture creation, innovation labs, business incubation, pitching.",
    "Leadership": "Leadership, management skills, organizational behavior, teamwork, decision-making, motivation.",
    "Volunteering": "Volunteering, social engagement, NGOs, community service, outreach, civic projects.",
    "StudentLife": "Student activities, clubs, associations, campus life, social events, orientation, parties.",
    "Sports": "Sports, athletics, tournaments, physical activity, competition, outdoor events, health initiatives.",
}

# HELPERS
def _csv_fingerprint(p: Path) -> str:
    stat = p.stat()
    meta = f"{p.resolve()}|{stat.st_mtime_ns}|{stat.st_size}"
    return hashlib.sha256(meta.encode("utf-8")).hexdigest()

def _texts_fingerprint(texts: list[str]) -> str:
    h = hashlib.sha256()
    for t in texts:
        h.update(b"\x00")
        h.update(t.encode("utf-8", errors="ignore"))
    return h.hexdigest()

def load_or_encode_events(model: SentenceTransformer, sentences: list[str], csv_path: Path, cache_path: Path) -> np.ndarray:
    """(Improvement #1) Stable cache keyed by model, csv fingerprint, and text hash."""
    key = {
        "model": MODEL_ID,
        "n": len(sentences),
        "csv_fp": _csv_fingerprint(csv_path),
        "texts_fp": _texts_fingerprint(sentences),
    }
    if cache_path.exists():
        try:
            payload = pickle.load(open(cache_path, "rb"))
            if payload.get("key") == key:
                return payload["vecs"]
        except Exception:
            pass
    vecs = np.array(model.encode(sentences, normalize_embeddings=True, show_progress_bar=False))
    pickle.dump({"key": key, "vecs": vecs}, open(cache_path, "wb"))
    return vecs

def build_event_text(row: pd.Series) -> str:
    """(#5) Weighted, tagged concatenation (title>subtitle>desc>speakers>loc)."""
    t  = str(row.get("source_subject") or "").strip()
    st = str(row.get("title") or "").strip()
    d  = str(row.get("description") or "").strip()
    sp = str(row.get("speakers") or "").strip()
    loc= str(row.get("location") or "").strip()
    parts = []
    if t:   parts += [f"source_subject: {t}"] * 4
    if st:  parts += [f"title: {st}"] * 2
    if d:   parts += [f"desc: {d}"]
    if sp:  parts += [f"speakers: {sp}"]
    if loc: parts += [f"loc: {loc}"]
    txt = " ".join(parts).lower()
    txt = re.sub(r"\s+", " ", txt).strip()
    return txt

def parse_event_dt(row: pd.Series) -> datetime | None:
    """(#3) TZ-aware parse of event_date + start_time; returns None if missing."""
    d = str(row.get("event_date") or "").strip()
    t = str(row.get("start_time") or "").strip()
    if not d:
        return None
    try:
        if t:
            dt = datetime.strptime(d + " " + t, "%Y-%m-%d %H:%M")
        else:
            dt = datetime.strptime(d, "%Y-%m-%d")
        return dt.replace(tzinfo=TZ)
    except Exception:
        return None

def time_decay(dt: datetime | None, ref: datetime, half_life_days: float = HALF_LIFE_DAYS) -> float:
    """(#3) Exponential decay toward the future date; past=0."""
    if dt is None:
        return 0.0
    delta_days = (dt - ref).total_seconds() / 86400.0
    if delta_days < 0:
        return 0.0
    return math.exp(-math.log(2) * (delta_days / half_life_days))

def mean_embed(model: SentenceTransformer, texts: list[str]) -> np.ndarray:
    V = model.encode(texts, normalize_embeddings=True, show_progress_bar=False)
    V = np.array(V)
    v = V.mean(axis=0)
    v = v / (np.linalg.norm(v) + 1e-12)
    return v

def mmr_indices(sim_to_q: np.ndarray, pairwise_sim: np.ndarray, k: int, lam: float) -> list[int]:
    """(#2) MMR on pure cosine for both relevance and diversity."""
    if len(sim_to_q) == 0:
        return []
    selected = [int(sim_to_q.argmax())]
    remaining = set(range(len(sim_to_q))) - set(selected)
    while len(selected) < min(k, len(sim_to_q)) and remaining:
        best_j, best_val = None, -1e18
        for j in remaining:
            max_sim = max(pairwise_sim[j, s] for s in selected) if selected else 0.0
            val = lam * sim_to_q[j] - (1 - lam) * max_sim
            if val > best_val:
                best_val, best_j = val, j
        selected.append(best_j)
        remaining.remove(best_j)
    return selected

def _normalize_01(x: np.ndarray) -> np.ndarray:
    """Safe [0,1] normalization."""
    if len(x) == 0:
        return x
    xmin, xmax = np.nanmin(x), np.nanmax(x)
    if not np.isfinite(xmin) or not np.isfinite(xmax) or xmax - xmin < 1e-12:
        return np.zeros_like(x)
    return (x - xmin) / (xmax - xmin)

def _is_cultural(text: str) -> bool:
    if not text:
        return False
    txt = str(text).lower()
    return any(re.search(rf"\b{re.escape(k)}\b", txt) for k in CULTURE_KEYS)


def _dedupe_by_title_date(df: pd.DataFrame) -> pd.DataFrame:
    """(#9) Remove near-duplicates by normalized title+date."""
    def key(row):
        t = re.sub(r"\W+", " ", str(row.get("source_subject") or "").lower()).strip()
        d = str(row.get("event_date") or "")
        return f"{t}|{d}"
    return df.loc[~df.apply(key, axis=1).duplicated()].copy()

# CORE API
def recommend_events(
    interests: list[str],
    interest_weights: dict[str, float] | None = None,  # e.g., {"AI":2.0, "Music":0.5}
    top_k: int = TOP_K,
    csv_in: Path = CSV_IN,
    csv_out: Path = CSV_OUT,
    use_cross_encoder: bool = True
) -> pd.DataFrame:
    """
    Returns a DataFrame and writes 'csv_out'.
    """

    # Load & basic prep (#8 guardrails)
    if not Path(csv_in).exists():
        raise FileNotFoundError(f"Events CSV not found: {csv_in}")

    df = pd.read_csv(csv_in)
    if df.empty:
        df.to_csv(csv_out, index=False, encoding="utf-8")
        return df

    # Build text & datetime
    df["__text__"] = df.apply(build_event_text, axis=1)
    df["__dt__"]   = df.apply(parse_event_dt, axis=1)

    # Auto-skip past events if configured (business rule)
    if AUTO_SKIP_PAST:
        now = datetime.now(TZ)
        df = df[df["__dt__"].apply(lambda x: (x is None) or (x >= now))].copy()
        if df.empty:
            df.to_csv(csv_out, index=False, encoding="utf-8")
            return df
        
    df = df.reset_index(drop=True)

    # Embeddings
    model = SentenceTransformer(MODEL_ID)
    event_vecs = load_or_encode_events(model, df["__text__"].tolist(), Path(csv_in), CACHE)

    # User vector (#6: interest weights + liked history blend)
    chosen = [i for i in interests if i in INTEREST_DESCRIPTIONS]
    if not chosen:
        raise ValueError("No valid interests selected. Choose from: " + ", ".join(INTEREST_DESCRIPTIONS.keys()))

    if interest_weights:
        texts = []
        for k, v in interest_weights.items():
            if k in INTEREST_DESCRIPTIONS and v > 0:
                texts += [INTEREST_DESCRIPTIONS[k]] * int(max(1, round(v * 2)))  # light weighting via repetition
        if not texts:
            texts = [INTEREST_DESCRIPTIONS[i] for i in chosen]
    else:
        texts = [INTEREST_DESCRIPTIONS[i] for i in chosen]

    user_vec = mean_embed(model, texts)  # interests core

    # Similarity & Freshness
    sims = event_vecs @ user_vec                          # cosine [-1,1]
    sims01 = (sims + 1.0) / 2.0                           # [0,1] for easier blending

    now = datetime.now(TZ)
    fresh = np.array([time_decay(dt, now) for dt in df["__dt__"]], dtype=float)
    if np.allclose(fresh, 0.0):
        fresh = np.ones_like(sims01)  # guardrail if all dates missing


    # Final blended score
    base_score = W_SIM * sims01 + W_TIME * fresh
    score = base_score

    # Candidate window & MMR (#2)
    cand = df.copy()
    cand["cosine"] = sims
    cand["fresh"]  = fresh
    cand["score"]  = score

    # WINDOW best by blended score
    win = cand.nlargest(min(WINDOW, len(cand)), "score")
    if win.empty:
        cand.to_csv(csv_out, index=False, encoding="utf-8")
        return cand

    # Optional cross-encoder rerank (Improvement #7 with budget)
    if _HAS_CROSS and use_cross_encoder and len(win) > 1:
        try:
            ce = CrossEncoder(CROSS_ID, max_length=256)
            # only rerank top-N best-scoring events for speed
            win_for_ce = win.nlargest(min(RERANK_TOP_N, len(win)), "score")
            idx_map = win_for_ce.index

            query = " | ".join([INTEREST_DESCRIPTIONS[i] for i in chosen])
            docs = (
                win_for_ce["source_subject"].fillna("").astype(str) + " | " +
                win_for_ce["title"].fillna("").astype(str) + " | " +
                win_for_ce["description"].fillna("").astype(str)
            ).tolist()
            pairs = [(query, d) for d in docs]

            ce_scores = ce.predict(pairs)  # higher = more relevant
            # store scores back in full 'win' DataFrame
            win.loc[idx_map, "ce_score"] = ce_scores
            # fill NaN with min to avoid NaN in blending
            win["ce_score"] = win["ce_score"].fillna(win["ce_score"].min(skipna=True) or 0.0)
            # blend reranker signal (20%) into total score
            win["score"] = 0.8 * win["score"] + 0.2 * _normalize_01(win["ce_score"].to_numpy())
        except Exception as e:
            # If reranker fails for any reason, proceed without it
            print("Cross-encoder rerank failed:", e)
            pass


    # Pairwise cosine among WINDOW
    Ew = event_vecs[win.index.values]
    Ew = Ew / (np.linalg.norm(Ew, axis=1, keepdims=True) + 1e-12)
    pairwise = Ew @ Ew.T

    # MMR on raw cosine
    sim_to_q = win["cosine"].to_numpy()
    idx = mmr_indices(sim_to_q, pairwise, k=min(top_k, len(win)), lam=MMR_LAMBDA)
    out = win.iloc[idx].copy()

    # Business rules (#9)
    out = _dedupe_by_title_date(out)

    # Cap per day
    if "event_date" in out.columns and CAP_PER_DAY > 0:
        out["_rank_day"] = out.groupby("event_date")["score"].rank(method="first", ascending=False)
        out = out[out["_rank_day"] <= CAP_PER_DAY].drop(columns=["_rank_day"])

    # Cap per organizer/source if present
    org_col = "source_from" if "source_from" in out.columns else ("organizer" if "organizer" in out.columns else None)
    if org_col and CAP_PER_ORGANIZER > 0:
        out["_rank_org"] = out.groupby(org_col)["score"].rank(method="first", ascending=False)
        out = out[out["_rank_org"] <= CAP_PER_ORGANIZER].drop(columns=["_rank_org"])

    # Try to include at least one cultural item if scores are close
    if REQUIRE_AT_LEAST_ONE_CULTURAL and len(out) > 0 and not out["__text__"].apply(_is_cultural).any():
        # look back into 'win' (already good candidates), find a cultural within 10% of min(top list) score
        min_kept = float(out["score"].min())
        candidates = win[win["__text__"].apply(_is_cultural)]
        if not candidates.empty:
            near = candidates[candidates["score"] >= 0.9 * min_kept]
            if not near.empty:
                # add the best cultural one
                add_row = near.nlargest(1, "score")
                out = pd.concat([out, add_row], ignore_index=True)
                # re-apply caps lightly
                if "event_date" in out.columns and CAP_PER_DAY > 0:
                    out["_rank_day"] = out.groupby("event_date")["score"].rank(method="first", ascending=False)
                    out = out[out["_rank_day"] <= CAP_PER_DAY].drop(columns=["_rank_day"])
                if org_col and CAP_PER_ORGANIZER > 0:
                    out["_rank_org"] = out.groupby(org_col)["score"].rank(method="first", ascending=False)
                    out = out[out["_rank_org"] <= CAP_PER_ORGANIZER].drop(columns=["_rank_org"])

    # Final display order: blended score -> date -> time
    out = out.sort_values(["score","event_date","start_time"], ascending=[False, True, True])

    # Observability (#10) ----
    # Provide transparent components per item
    out["explain_cosine01"] = (out["cosine"] + 1.0) / 2.0
    out["explain_fresh"]    = out["fresh"]
    out["explain_base"]     = W_SIM * out["explain_cosine01"] + W_TIME * out["explain_fresh"]
    out["explain_score"]    = out["score"]

    # Ensure keep columns exist
    keep = ["event_date","start_time","end_time","source_subject","title","description","speakers","location","url",
            "cosine","fresh","score","explain_cosine01","explain_fresh","explain_base","explain_score"]
    for c in keep:
        if c not in out.columns:
            out[c] = ""
    out = out[keep]

    out.to_csv(csv_out, index=False, encoding="utf-8")
    return out

# CLI DEMO 
if __name__ == "__main__":
    demo_interests = ["AI", "Data", "Sustainability", "Music"]
    recs = recommend_events(
        interests=demo_interests,
        interest_weights={"AI": 2.0, "Data": 1.5, "Music": 0.7},  # optional                                    # optional
        top_k=TOP_K,
        csv_in=CSV_IN,
        csv_out=CSV_OUT,
        use_cross_encoder=True
    )
    print("Saved ->", CSV_OUT.resolve())
    cols = ["event_date","start_time","source_subject","title","score","cosine","fresh"]
    print(recs[cols].head(TOP_K).to_string(index=False))

Saved -> C:\Users\sovai\OneDrive\DSP\recommendations.csv
event_date start_time                                                               source_subject                                       title    score   cosine    fresh
2026-05-13      20:00    Semestereröffnungskonzert am kommenden Donnerstag, 23.10.2025 im Festsaal    Christoph und Julian Prégardien (Tenöre) 0.625821 0.182824 0.000058
2026-01-11      20:00    Semestereröffnungskonzert am kommenden Donnerstag, 23.10.2025 im Festsaal Sinfonieorchester des Nationaltheaters Prag 0.602990 0.186725 0.024107
2025-10-28      16:00 Einladung: 25 Jahre Brasilien- und Lateinamerika-Zentrum – Jubiläumsprogramm                                         NaN 0.506991 0.186908 0.996303
2025-10-29      15:00 Einladung: 25 Jahre Brasilien- und Lateinamerika-Zentrum – Jubiläumsprogramm                                         NaN 0.503298 0.186908 0.950135
